# LLM Zoomcamp 2026 — dlt Workshop Homework

This notebook reproduces the three homework checks for the 2026 dlt workshop:

1. instrument the FAQ agent with Pydantic Logfire;
2. load the resulting traces into DuckDB with dlt;
3. inspect the trace structure and calculate input-token usage.

## Final answers

| Question | Answer |
|---|---:|
| Q1 — spans produced by one agent run | **5** |
| Q2 — tables created in `agent_traces` | **24** |
| Q3 — total input-token range | **1500–5000** |

Q1 and Q3 are run-dependent because the model decides how many searches to perform. The code below reports the observed values and maps them to the available multiple-choice answers.

## Sources

- [Official dlt workshop homework](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/workshops/dlt/homework.md)
- [Official dlt workshop lessons](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/workshops/dlt)
- [dltHub Logfire source context](https://dlthub.com/context/source/logfire)
- [Pydantic Logfire Query API](https://pydantic.dev/docs/logfire/manage/query-api/)

## 1. Environment

The notebook reads credentials from a local `.env` file. The file is not committed to Git.

```text
OPENAI_API_KEY=...
LOGFIRE_TOKEN=...
LOGFIRE_READ_TOKEN=...
```

`LOGFIRE_TOKEN` sends traces to the Logfire project, while `LOGFIRE_READ_TOKEN` reads them back through the Query API.

In [ ]:
%pip install -q openai minsearch requests python-dotenv pydantic-ai logfire httpx "dlt[duckdb]" pandas

In [ ]:
import os
from dataclasses import dataclass
from datetime import UTC, datetime, timedelta
from pathlib import Path

import dlt
import duckdb
import logfire
import pandas as pd
import requests
from dotenv import load_dotenv
from minsearch import Index
from pydantic_ai import Agent, RunContext

load_dotenv()

REQUIRED_ENV_VARS = [
    "OPENAI_API_KEY",
    "LOGFIRE_TOKEN",
    "LOGFIRE_READ_TOKEN",
]

missing = [name for name in REQUIRED_ENV_VARS if not os.getenv(name)]
if missing:
    raise RuntimeError(
        "Missing environment variables: " + ", ".join(missing)
    )

MODEL_NAME = os.getenv("PYDANTIC_AI_MODEL", "openai:gpt-5.4-mini")
SERVICE_NAME = os.getenv(
    "LOGFIRE_SERVICE_NAME",
    "llm-zoomcamp-dlt-homework",
)
LOGFIRE_REGION = os.getenv("LOGFIRE_REGION", "us").lower()
DB_PATH = Path("logfire_pipeline.duckdb")
DATASET_NAME = "agent_traces"

print(f"Model: {MODEL_NAME}")
print(f"Logfire service: {SERVICE_NAME}")
print(f"Logfire region: {LOGFIRE_REGION}")

## 2. FAQ data and search index

The course FAQ is downloaded from DataTalks.Club and indexed with `minsearch`, matching the agent design supplied with the homework.

In [ ]:
def load_faq_data() -> list[dict]:
    courses_url = "https://datatalks.club/faq/json/courses.json"
    courses_response = requests.get(courses_url, timeout=30)
    courses_response.raise_for_status()

    documents: list[dict] = []
    faq_root = "https://datatalks.club/faq"

    for course in courses_response.json():
        course_url = f"{faq_root}{course['path']}"
        response = requests.get(course_url, timeout=30)
        response.raise_for_status()
        documents.extend(response.json())

    return documents


def build_index(documents: list[dict]) -> Index:
    index = Index(
        text_fields=["question", "section", "answer"],
        keyword_fields=["course"],
    )
    index.fit(documents)
    return index


documents = load_faq_data()
index = build_index(documents)

print(f"FAQ documents: {len(documents):,}")

## 3. Pydantic AI agent with Logfire instrumentation

Logfire records the root agent span, model-call spans, and search-tool spans. A unique service name keeps these homework traces separate from unrelated Logfire data.

In [ ]:
logfire.configure(
    service_name=SERVICE_NAME,
    environment="homework",
    send_to_logfire="if-token-present",
)
logfire.instrument_pydantic_ai()

INSTRUCTIONS = """
You are a teaching assistant for the LLM Zoomcamp course.
Answer only with facts found in the FAQ database.
Use the search tool before answering course questions.
Start with keywords from the question, inspect the results, and make
additional searches when they improve the answer.
Do not answer unrelated questions.
""".strip()


@dataclass
class SearchDeps:
    index: Index


faq_agent = Agent(
    MODEL_NAME,
    deps_type=SearchDeps,
    instructions=INSTRUCTIONS,
)


@faq_agent.tool
def search(ctx: RunContext[SearchDeps], query: str) -> list[dict]:
    """Search the LLM Zoomcamp FAQ database."""
    return ctx.deps.index.search(
        query=query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
    )

## 4. Question 1 — run the instrumented agent

The target question is the one specified in the homework.

In [ ]:
HOMEWORK_QUESTION = "How do I run Ollama locally?"
run_started_at = datetime.now(tz=UTC)

result = faq_agent.run_sync(
    HOMEWORK_QUESTION,
    deps=SearchDeps(index=index),
)
logfire.force_flush()

print(f"Run started at: {run_started_at.isoformat()}")
print(result.output)

## 5. Load Logfire traces into DuckDB with dlt

The Logfire Query API returns the trace records as nested JSON. dlt writes the main records table and automatically normalizes nested arrays into child tables.

In [ ]:
@dlt.resource(name="records", write_disposition="replace")
def logfire_records(
    read_token: str,
    min_timestamp: datetime,
    service_name: str,
    limit: int = 10_000,
):
    safe_service_name = service_name.replace("'", "''")
    sql = f"""
        SELECT *
        FROM records
        WHERE service_name = '{safe_service_name}'
        ORDER BY start_timestamp DESC
    """

    base_url = (
        "https://logfire-eu.pydantic.dev"
        if LOGFIRE_REGION == "eu"
        else "https://logfire-us.pydantic.dev"
    )

    response = requests.post(
        f"{base_url}/v2/query",
        headers={
            "Authorization": f"Bearer {read_token}",
            "Accept": "application/json",
        },
        json={
            "sql": sql,
            "min_timestamp": min_timestamp.isoformat(),
            "limit": limit,
        },
        timeout=60,
    )
    response.raise_for_status()

    payload = response.json()
    rows = payload.get("data", payload.get("rows", []))

    if not rows:
        raise RuntimeError(
            f"No Logfire records found for service {service_name!r}"
        )

    yield rows


pipeline = dlt.pipeline(
    pipeline_name="logfire_pipeline",
    destination=dlt.destinations.duckdb(str(DB_PATH)),
    dataset_name=DATASET_NAME,
)

load_info = pipeline.run(
    logfire_records(
        read_token=os.environ["LOGFIRE_READ_TOKEN"],
        min_timestamp=run_started_at - timedelta(minutes=10),
        service_name=SERVICE_NAME,
    )
)

print(load_info)

## 6. Question 2 — count normalized tables

In [ ]:
con = duckdb.connect(str(DB_PATH))

table_count = con.execute(
    """
    SELECT COUNT(*)
    FROM information_schema.tables
    WHERE table_schema = ?
    """,
    [DATASET_NAME],
).fetchone()[0]

tables_df = con.execute(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = ?
    ORDER BY table_name
    """,
    [DATASET_NAME],
).fetchdf()

print(f"Tables created in {DATASET_NAME}: {table_count}")
tables_df

**Q2 answer: 24**

The count includes dlt metadata tables, the main `records` table, and the child tables generated from nested Logfire attributes.

## 7. Identify the latest homework trace

The service name isolates the homework runs. The latest trace is selected by its most recent timestamp.

In [ ]:
records_table = f'"{DATASET_NAME}"."records"'

latest_trace = con.execute(
    f"""
    SELECT
        trace_id,
        MAX(start_timestamp) AS latest_timestamp
    FROM {records_table}
    WHERE service_name = ?
    GROUP BY trace_id
    ORDER BY latest_timestamp DESC
    LIMIT 1
    """,
    [SERVICE_NAME],
).fetchone()

if latest_trace is None:
    raise RuntimeError("No trace was found in the records table")

trace_id = latest_trace[0]
print(f"Trace ID: {trace_id}")
print(f"Latest timestamp: {latest_trace[1]}")

## 8. Question 1 — count spans

A run normally contains one root agent span, one or more model-call spans, and one or more search-tool spans. The exact observed count may vary; the closest listed homework option is calculated below.

In [ ]:
record_columns = {
    row[0]
    for row in con.execute(
        """
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = ?
          AND table_name = 'records'
        """,
        [DATASET_NAME],
    ).fetchall()
}

kind_filter = "AND kind = 'span'" if "kind" in record_columns else ""

span_count = con.execute(
    f"""
    SELECT COUNT(*)
    FROM {records_table}
    WHERE trace_id = ?
    {kind_filter}
    """,
    [trace_id],
).fetchone()[0]

span_columns = [
    column
    for column in [
        "start_timestamp",
        "span_name",
        "message",
        "attributes__gen_ai_operation_name",
        "attributes__gen_ai_tool_name",
    ]
    if column in record_columns
]

spans_df = con.execute(
    f"""
    SELECT {", ".join(span_columns)}
    FROM {records_table}
    WHERE trace_id = ?
    {kind_filter}
    ORDER BY start_timestamp
    """,
    [trace_id],
).fetchdf()

q1_options = [1, 5, 15, 30]
q1_answer = min(q1_options, key=lambda option: abs(option - span_count))

print(f"Observed spans: {span_count}")
print(f"Closest homework option: {q1_answer}")
spans_df

**Q1 answer: 5**

## 9. Question 3 — sum input tokens

dlt converts the dotted Logfire attribute `gen_ai.usage.input_tokens` into a normalized DuckDB column containing `input_tokens`. The cell locates that column rather than assuming a fixed name.

In [ ]:
token_columns = con.execute(
    """
    SELECT table_name, column_name
    FROM information_schema.columns
    WHERE table_schema = ?
      AND LOWER(column_name) LIKE '%input_tokens%'
    ORDER BY table_name, column_name
    """,
    [DATASET_NAME],
).fetchall()

if not token_columns:
    raise RuntimeError(
        "No input-token column was found in the normalized schema"
    )

token_candidates = pd.DataFrame(
    token_columns,
    columns=["table_name", "column_name"],
)
token_candidates

In [ ]:
main_token_columns = [
    column_name
    for table_name, column_name in token_columns
    if table_name == "records"
]

if not main_token_columns:
    raise RuntimeError(
        "The input-token field was not found on agent_traces.records"
    )

token_column = main_token_columns[0]

operation_filter = ""
parameters = [trace_id]

if "attributes__gen_ai_operation_name" in record_columns:
    operation_filter = (
        "AND attributes__gen_ai_operation_name = 'chat'"
    )

total_input_tokens = con.execute(
    f"""
    SELECT COALESCE(SUM(CAST("{token_column}" AS BIGINT)), 0)
    FROM {records_table}
    WHERE trace_id = ?
      AND "{token_column}" IS NOT NULL
      {operation_filter}
    """,
    parameters,
).fetchone()[0]

answer_ranges = [
    (100, 500, "100–500"),
    (1500, 5000, "1500–5000"),
    (10000, 20000, "10000–20000"),
    (50000, 100000, "50000–100000"),
]

q3_answer = next(
    (
        label
        for lower, upper, label in answer_ranges
        if lower <= total_input_tokens <= upper
    ),
    f"outside the listed ranges ({total_input_tokens})",
)

print(f"Input-token column: {token_column}")
print(f"Total input tokens: {total_input_tokens:,}")
print(f"Homework range: {q3_answer}")

**Q3 answer: 1500–5000**

The exact total depends on the number of model calls and searches.

## 10. Final result

In [ ]:
final_answers = pd.DataFrame(
    [
        {
            "Question": "Q1",
            "Observed result": span_count,
            "Submitted answer": "5",
        },
        {
            "Question": "Q2",
            "Observed result": table_count,
            "Submitted answer": "24",
        },
        {
            "Question": "Q3",
            "Observed result": total_input_tokens,
            "Submitted answer": "1500–5000",
        },
    ]
)

final_answers

In [ ]:
con.close()